# Javtiful → GoFile Telegram Bot

This dedicated notebook runs the `gofile` branch only. It needs **only two Colab Secrets**: `BOT_TOKEN` and `OWNER_ID`. No `API_ID`, `API_HASH`, database, Neon, or GoFile token is required.

1. Add `BOT_TOKEN` and `OWNER_ID` in Colab Secrets and enable notebook access.
2. Run Cell 1 to install dependencies and fetch the latest `gofile` branch.
3. Run Cell 2 to compile the bot and start polling. Stop the cell to stop the bot.

Send a video code such as `STAR-765`. Search sessions are kept in memory only. Javtiful's website and playback format may change, so verify live search/download behaviour after startup.

In [ ]:
# Cell 1 — Load the two secrets, install tools, and fetch the gofile branch
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

required = ("BOT_TOKEN", "OWNER_ID")
for key in required:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value is None or not str(value).strip():
        raise RuntimeError(f"Missing Colab Secret: {key}. Add it in Secrets and enable notebook access.")
    os.environ[key] = str(value).strip()

try:
    owner_id = int(os.environ["OWNER_ID"])
    if owner_id <= 0:
        raise ValueError
except ValueError as exc:
    raise ValueError("OWNER_ID must be a positive numeric Telegram user ID.") from exc

REPO_DIR = Path("/content/URL-uploader-gofile")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "gofile", "https://github.com/Telebotfaroff/URL-uploader.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_gofile.txt"], check=True)
print("Setup complete. Secrets loaded without displaying their values.")

In [ ]:
# Cell 2 — Compile the bot and start it
import os
import subprocess
import sys

os.environ["BOT_TOKEN"] = str(userdata.get("BOT_TOKEN")).strip()
os.environ["OWNER_ID"] = str(userdata.get("OWNER_ID")).strip()
subprocess.run([sys.executable, "-m", "compileall", "-q", "gofile_bot.py", "gofile_search.py", "gofile_media.py", "gofile_uploader.py", "plugins/javdl_core"], check=True)
print("Source compilation passed. Starting owner-only bot…")
subprocess.run([sys.executable, "gofile_bot.py"], check=True)